# Research Question 4

### *What are the assumptions required to address the limited observability issues?* 

*Ideas proposed in the literature review.*

In order to address this issue of observability for DGs, as discussed in Section 3-3 (Literature Review), the most dominant theme is the generation of pseudo-measurements. For the open-source test case networks, a Gaussian distribution can be a probable assumption for generating synthetic data. For practical DGs, topological observability is required, assuming a minimal spanning tree exists in the network. However, if the network is not observable and, 

- if there are few available measurements, then these measurements can be utilized to design a probabilistic dependency between them [25], which can be utilized in generating pseudo-measurements as unobservable nodes. Moreover, the high sampling rate of the measurement devices can be considered for accurate interpolation of the pseudomeasurements. 

- if no measurements are available, then it can be proposed to employ a Gaussian mixture models to devise a joint distribution of measurements for generating the pseudo-measurements. 

*Ideas that will be tested here.*

1. Generate Pseudo-measurements as high standard deviation noise on power flow results and measurements as low standard deviation noise on power flow results. Then, tune the noise such that WLS converges. Compare WLS performance with GNN 4 SE. 

(Extra: Devise state-vector augmentation and method of residual analysis to perform conventional parameter estimation.)

2. GANs. See (`rq_4_gans.ipynb`)



### Methodology 

In [2]:
import pandapower as pp 
# from utils import custom_se
import os 
import sys 
import numpy as np 
import joblib
from ipywidgets import widgets


# for complex numbers printing
np.set_printoptions(formatter={'complex_kind': '{:.2f}'.format})

# Get the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
print(parent_dir)
sys.path.insert(0, parent_dir)

from utils.ppnet_utils import  drop_pf_results, initialize_network

/Users/sohamprajapati/Documents/Thesis/Project Codes/TapSEGNN


In [ ]:
def custom_se_rq4(net: pp.pandapowerNet, 
                  std_meas: float = 0.01, 
                  std_pseudo: float = 0.02,
                  sparsity_prob: float = 0.5, 
                  se_iter: int = 10):
    """

    Bus_type_1: With measurements 
    Bus_type_2: No measurements 
    Noise on measurements at bus_type_1: power flow results + std_meas 
    Noise on measurements at bus_type_2: power flow results + std_pseudo

    Args: 
        net (pp.pandapowerNet): Pandapower Network.
        std_meas (float): standard deviation of zero-mean Gaussian noise over the buses. 
        std_pseudo (float): standard deviation of zero-mean Gaussian noise over the buses. 

    Returns: 
        net_meas (pp.pandapowerNet): Net with state-estimation results if converged else original net.

    """

    # drop the measurements or power flow results if any 
    net.measurement.drop(net.measurement.index, inplace = True)
    net = drop_pf_results(net)

    # perform power flow 
    pp.runpp(net)

    # for at bus measurements
    vm_pfr = net.res_bus.vm_pu
    p_pfr = net.res_bus.p_mw 
    
    # for to/from bus measurements 
    p_pfr_from_line = net.res_line.p_from_mw
    q_pfr_from_line = net.res_line.q_from_mvar
    p_pfr_to_line = net.res_line.p_to_mw
    q_pfr_to_line = net.res_line.q_to_mvar

    # all bus indices
    all_bus = np.array(net.bus.index)

    if net.name == "net_A": 
        
        # v, p measurements available at buses 
        at_bus_meas = np.array([0,1,2,3,5,7,11,17,22])

        # p_flow, q_flow measuremnets available at switching stations to and from the bus 
        flow_bus_meas = np.array([5,7,11,17,22])
        
    else: 
        # consider at and flow buses same for other networks 
        # with some sparsity probability 
        meas_bus_mask = np.random.rand(all_bus.size) >= sparsity_prob 
        at_bus_meas = np.array([bus_id for bus_id in all_bus if meas_bus_mask[bus_id]])

        flow_bus_meas = at_bus_meas 

    # ~at_bus_meas 
    not_at_bus_mask = np.ones(all_bus.size, dtype=bool)
    not_at_bus_mask[at_bus_meas] = False
    not_at_bus_meas = all_bus[not_at_bus_mask]

    not_flow_bus_mask = np.ones(all_bus.size, dtype=bool)
    not_flow_bus_mask[flow_bus_meas] = False 
    not_flow_bus_meas = all_bus[not_flow_bus_mask]

    num_meas = 0
    # create v, p measurements at at_bus_meas 
    for idx, (idx_bus) in enumerate(at_bus_meas):
        num_meas += 2
        vm_at_idx_bus = vm_pfr[idx] + np.random.normal(0, std_meas)
        pmw_at_idx_bus = p_pfr[idx] + np.random.normal(0, std_meas)

        pp.create.create_measurement(net, "v", "bus", value=vm_at_idx_bus, std_dev=std_meas, element = idx_bus)
        pp.create.create_measurement(net, "p", "bus", value=pmw_at_idx_bus, std_dev=std_meas, element = idx_bus)
    
    # create v, p measurements at not_at_bus_meas 
    for idx, (idx_bus) in enumerate(not_at_bus_meas):
        num_meas += 2
        vm_at_idx_bus = vm_pfr[idx] + np.random.normal(0, std_pseudo)
        pmw_at_idx_bus = p_pfr[idx] + np.random.normal(0, std_pseudo)

        pp.create.create_measurement(net, "v", "bus", value=vm_at_idx_bus, std_dev=std_pseudo, element = idx_bus)
        pp.create.create_measurement(net, "p", "bus", value=pmw_at_idx_bus, std_dev=std_pseudo, element = idx_bus)


    # create p_from, q_from, p_to, q_to measurements at flow_bus_meas 
    for iline, line in net.line.iterrows(): 
        from_bus = int(line.from_bus)
        to_bus = int(line.to_bus)

        if to_bus in flow_bus_meas: 
            num_meas += 2
            p_to_mw_meas = p_pfr_to_line[iline] + np.random.normal(0, std_meas)
            q_to_mvar_meas = q_pfr_to_line[iline] + np.random.normal(0, std_meas)
            pp.create.create_measurement(net, "p", "line", value=p_to_mw_meas, std_dev=std_meas, side="to", element=iline)
            pp.create.create_measurement(net, "q", "line", value=q_to_mvar_meas, std_dev=std_meas, side="to", element=iline)

        if from_bus in flow_bus_meas: 
            num_meas += 2
            p_from_mw_meas = p_pfr_from_line[iline] + np.random.normal(0, std_meas)
            q_from_mvar_meas = q_pfr_from_line[iline] + np.random.normal(0, std_meas)
            pp.create.create_measurement(net, "p", "line", value=p_from_mw_meas, std_dev=std_meas, side="from", element=iline)
            pp.create.create_measurement(net, "q", "line", value=q_from_mvar_meas, std_dev=std_meas, side="from", element=iline)

        if to_bus in not_flow_bus_meas:
            num_meas += 2
            p_to_mw_meas = p_pfr_to_line[iline] + np.random.normal(0, std_pseudo)
            q_to_mvar_meas = q_pfr_to_line[iline] + np.random.normal(0, std_pseudo)
            pp.create.create_measurement(net, "p", "line", value=p_to_mw_meas, std_dev=std_pseudo, side="to", element=iline)
            pp.create.create_measurement(net, "q", "line", value=q_to_mvar_meas, std_dev=std_pseudo, side="to", element=iline)

        if from_bus in not_flow_bus_meas: 
            num_meas += 2
            p_from_mw_meas = p_pfr_from_line[iline] + np.random.normal(0, std_pseudo)
            q_from_mvar_meas = q_pfr_from_line[iline] + np.random.normal(0, std_pseudo)
            pp.create.create_measurement(net, "p", "line", value=p_from_mw_meas, std_dev=std_pseudo, side="from", element=iline)
            pp.create.create_measurement(net, "q", "line", value=q_from_mvar_meas, std_dev=std_pseudo, side="from", element=iline)  

    # drop the power flow results 
    net = drop_pf_results(net)

    # number of measurements 
    n_bus = len(net.bus.index)
    n_meas = len(net.bus.index)*2 + len(net.line.index)*4 
    print(f"n_meas = {n_meas}")

    success = pp.estimation.estimate(net, algorithm="wls", init="flat")

    if success: 
        print(f"State estimation successful for {net.name}!")
        # number of measurements 
        print(f"Total number of measurements = {n_meas} and number of measurements should be at least {2*n_bus - 1}")
        
        return net
    elif not success and se_iter > 1:
        # if verbose:
        print(f"State estimation failed. Retrying... Remaining attempts: {se_iter - 1}")
        return custom_se_rq4(net=net, 
                             std_meas=std_meas, 
                             std_pseudo=std_pseudo, 
                             sparsity_prob=sparsity_prob, 
                             se_iter=se_iter - 1) 
    
    else: 
        UserWarning(f"Solver failed for {net.name}, returning net as it is.")
        # number of measurements 
        print(f"Total number of measurements = {n_meas} and number of measurements should be at least {2*n_bus - 1}")
        return net

net = initialize_network(net_name='net_A',
                         load_std=0.01)
net_meas = custom_se_rq4(net=net, std_meas=0.01, std_pseudo=0.02, sparsity_prob=0.0, se_iter=10)

Network: net_A is selected 

Net net_A has 42 nodes and 42 edges. 



DEBUG:pandapower.estimation.state_estimation:State Estimation successful (6 iterations)
DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=113)
           2	RESUME(arg=0, lineno=113)
           4	LOAD_GLOBAL(arg=1, lineno=116)
          16	LOAD_FAST(arg=5, lineno=116)
          18	LOAD_GLOBAL(arg=2, lineno=116)
          30	KW_NAMES(arg=1, lineno=116)
          32	PRECALL(arg=2, lineno=116)
          36	CALL(arg=2, lineno=116)
          46	STORE_FAST(arg=7, lineno=116)
          48	LOAD_GLOBAL(arg=5, lineno=119)
          60	LOAD_GLOBAL(arg=7, lineno=119)
          72	LOAD_FAST(arg=1, lineno=119)
          74	PRECALL(arg=1, lineno=119)
          78	CALL(arg=1, lineno=119)
          88	LOAD_CONST(arg=2, lineno=119)
          90	BINARY_OP(arg=10, lineno=119)
          94	PRECALL(arg=1, lineno=119)
          98	CALL(arg=1, lineno=119)
         108	GET_ITER(arg=None, lineno=119)
>        110	FOR_ITER(arg=89, lineno=119)
         112	STORE_FAST(arg=8, lineno=119)
   

n_meas = 172


DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=4291)
           2	RESUME(arg=0, lineno=4291)
           4	LOAD_GLOBAL(arg=1, lineno=4292)
          16	LOAD_FAST(arg=1, lineno=4292)
          18	LOAD_FAST(arg=2, lineno=4292)
          20	PRECALL(arg=2, lineno=4292)
          24	CALL(arg=2, lineno=4292)
          34	RETURN_VALUE(arg=None, lineno=4292)
DEBUG:numba.core.byteflow:pending: deque([State(pc_initial=0 nstack_initial=0)])
DEBUG:numba.core.byteflow:stack: []
DEBUG:numba.core.byteflow:state.pc_initial: State(pc_initial=0 nstack_initial=0)
DEBUG:numba.core.byteflow:dispatch pc=0, inst=NOP(arg=None, lineno=4291)
DEBUG:numba.core.byteflow:stack []
DEBUG:numba.core.byteflow:dispatch pc=2, inst=RESUME(arg=0, lineno=4291)
DEBUG:numba.core.byteflow:stack []
DEBUG:numba.core.byteflow:dispatch pc=4, inst=LOAD_GLOBAL(arg=1, lineno=4292)
DEBUG:numba.core.byteflow:stack []
DEBUG:numba.core.byteflow:dispatch pc=16, inst=LOAD_FAST(arg=1, lineno=4292)
DEBUG:numba.core

State estimation successful for !
Total number of measurements = 172 and number of measurements should be at least 83


In [4]:
pp.runpp(net)
vm_pfr = net.res_bus.vm_pu
va_pfr = net.res_bus.va_degree

# calculate RMSE values of state-estimates 
rmse_mag_v = float(np.sqrt(np.mean((vm_pfr - np.array(net_meas.res_bus_est.vm_pu))**2))) 
rmse_mag_a = float(np.sqrt(np.mean((va_pfr - np.array(net_meas.res_bus_est.va_degree))**2)))

DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=58)
           2	RESUME(arg=0, lineno=58)
           4	LOAD_GLOBAL(arg=1, lineno=61)
          16	LOAD_GLOBAL(arg=3, lineno=61)
          28	LOAD_FAST(arg=1, lineno=61)
          30	PRECALL(arg=1, lineno=61)
          34	CALL(arg=1, lineno=61)
          44	PRECALL(arg=1, lineno=61)
          48	CALL(arg=1, lineno=61)
          58	GET_ITER(arg=None, lineno=61)
>         60	FOR_ITER(arg=83, lineno=61)
          62	STORE_FAST(arg=10, lineno=61)
          64	LOAD_FAST(arg=4, lineno=62)
          66	LOAD_FAST(arg=10, lineno=62)
          68	BINARY_SUBSCR(arg=None, lineno=62)
          78	POP_JUMP_FORWARD_IF_FALSE(arg=20, lineno=62)
          80	LOAD_FAST(arg=3, lineno=62)
          82	LOAD_FAST(arg=10, lineno=62)
          84	BINARY_SUBSCR(arg=None, lineno=62)
          94	POP_JUMP_FORWARD_IF_FALSE(arg=12, lineno=62)
          96	LOAD_FAST(arg=5, lineno=62)
          98	LOAD_FAST(arg=10, lineno=62)
         100	BINAR

In [6]:
rmse_mag_v, rmse_mag_a

(0.007864020907787848, 0.11881386906993004)